# 02 - First milestone: one MODIS LST map per city

Fetch a single MOD11A2 composite, save it as a raw GeoTIFF with a catalogue row, quality-filter and scale it, and look at both cities. Do this before downloading 25 years of data (SRS FR-P2.9).

Needs `grid.json` and the boundaries from notebook 01.

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/Starpredetor/urban-heat-risk-prediction.git'
REPO_DIR = '/content/uhrm'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR + '/src')
from uhrm.colab import bootstrap
project = bootstrap()

In [ ]:
from datetime import timedelta
import numpy as np
import matplotlib.pyplot as plt
from uhrm import acquire, modis, regions
from uhrm.backends.earthengine import EarthEngineModis

grid = regions.load_grid(project.path('grid.json'))
geoms = {a.slug: regions.load_boundary(project.boundary_path(a), project.grid.crs) for a in project.study_areas}
masks = {s: grid.cell_mask(g, project.grid.min_cell_overlap) for s, g in geoms.items()}
backend = EarthEngineModis(project.modis.collection)

start = project.modis.first_milestone_date
dates = backend.list_composites(start, start + timedelta(days=40))
print('composites found:', dates)
day = dates[0]

## Fetch and save the raw composite
Written once, never modified (FR-P2.6). Re-running skips it. This is the first real call to `computePixels`, so an error here is useful feedback about the Earth Engine backend.

In [ ]:
out_dir = project.path('raw', 'modis')
summary = acquire.acquire_modis(backend, grid, [day], out_dir, project.path('catalog.csv'))
print(summary)

## Quality-filter, scale to Celsius, inspect

In [ ]:
import rasterio
with rasterio.open(acquire.composite_path(out_dir, day)) as src:
    dn, qc = src.read(1), src.read(2)
lst = modis.process_composite(dn, qc, project.modis.qc)
print(f'whole domain: {np.isfinite(lst).mean():.0%} of cells valid')
for slug, m in masks.items():
    v = lst[m]
    ok = np.isfinite(v)
    if ok.any():
        print(f'{slug}: {m.sum()} cells, {ok.mean():.0%} valid, LST {v[ok].min():.1f} to {v[ok].max():.1f} C, mean {v[ok].mean():.1f} C')
    else:
        print(f'{slug}: {m.sum()} cells, none valid on this date')

In [ ]:
x0, y0, x1, y1 = grid.bounds
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
im = axes[0].imshow(lst, extent=(x0, x1, y0, y1), cmap='inferno')
axes[0].set_title(f'MOD11A2 daytime LST (C), composite starting {day}')
fig.colorbar(im, ax=axes[0], shrink=0.7)
axes[1].imshow(np.isfinite(lst), extent=(x0, x1, y0, y1), cmap='gray')
axes[1].set_title('valid cells (white) after quality filter')
for ax in axes:
    for g in geoms.values():
        for poly in getattr(g, 'geoms', [g]):
            ax.plot(*poly.exterior.xy, 'c-', lw=0.8)
plt.tight_layout()
plt.show()

## Milestone checklist (SRS section 1.7)
- April daytime LST is in a plausible range for land (roughly 30-50 C); the sea is cooler than the land.
- The coastline and both outlines line up with the temperature field.
- Cells the filter rejected are where you would expect cloud or coast, not a random pattern.
- Both cities show some valid cells. If one has none on this date, try another composite before concluding anything.
- Open the saved GeoTIFF in QGIS (after downloading it from Drive) and confirm it lines up with the boundary; that is the final check for this milestone.